# Step 3 — Reproject a band to the analysis grid

Resamples one band from its UTM grid onto a regular latitude/longitude grid covering the
study area (EPSG:4326, 0.0001° ≈ 10 m, nearest neighbour), the grid used by the
single-notebook version through `stackstac.stack(epsg=4326, resolution=0.0001)`.
The workflow runs this step once per band.

| | |
|---|---|
| Six-phase position | Pre-processing |
| W1 Algae Bloom counterpart | `reproject-image` |
| Output | `reprojected_file`: `<input name>_<epsg>.tif` |

Unlike `reproject-image` (resampling only, no CRS change), this step does change the CRS.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "ResourceRequirement": {"coresMin": 1, "ramMin": 1024},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "reprojection"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Reproject a raster onto a regular grid covering a bounding box",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
band_file: CWLFilePathInput = "red.tif"
west: CWLFloatInput = 95.15
south: CWLFloatInput = 15.9
east: CWLFloatInput = 95.35
north: CWLFloatInput = 16.1
epsg: Optional[CWLIntInput] = 4326
resolution: Optional[CWLFloatInput] = 0.0001

## Target grid

In [ ]:
# snap the study area on the resolution so that every band gets the same grid
left = np.floor(west / resolution) * resolution
top = np.ceil(north / resolution) * resolution
width = int(np.ceil((east - left) / resolution))
height = int(np.ceil((top - south) / resolution))
dst_transform = from_origin(left, top, resolution, resolution)
print(f"Target grid: EPSG:{epsg}, {width} x {height} px at {resolution}")

## Reproject

In [ ]:
with rasterio.open(band_file) as src:
    destination = np.full((height, width), np.nan, dtype="float32")
    reproject(
        source=rasterio.band(src, 1),
        destination=destination,
        src_nodata=np.nan,
        dst_transform=dst_transform,
        dst_crs=f"EPSG:{epsg}",
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )
    tags = src.tags(1)

reprojected_file: CWLFilePathOutput = f"{Path(band_file).stem}_{epsg}.tif"
with rasterio.open(
    reprojected_file,
    "w",
    driver="GTiff",
    width=width,
    height=height,
    count=1,
    dtype="float32",
    crs=f"EPSG:{epsg}",
    transform=dst_transform,
    nodata=np.nan,
    compress="deflate",
) as dst:
    dst.write(destination, 1)
    dst.update_tags(1, **tags)
print(f"Saved: {reprojected_file}")